# 02 — Repeated stratified cross-validation (all datasets, all models)

This notebook is the provenance of **Table 6** (cross-validation, all datasets) and of the
paired model comparison used in the results discussion. It runs, or reloads,
a `5 x 5` repeated stratified cross-validation for every dataset and every
model in the paper.

## Fold construction (identical for every model)

For repeat `r` and fold `k`:

```python
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=1000 + r)
pool, test = outer.split(X, y)                    # 800 / 200 on German Credit
train, val = train_test_split(pool, test_size=0.2,
                              stratify=y[pool],
                              random_state=2000 + 10*r + k)   # 640 / 160
```

So each fold exposes `640 / 160 / 200` examples for German Credit, i.e. the
inner split keeps a validation partition for early stopping while the outer
20% stays untouched until scoring. **The same indices are used for SCORE-Net
and for every baseline**, which makes the comparison paired.

SCORE-Net fits on `train` and early-stops on `val`. Baselines have frozen
hyperparameters, so they fit on `train` and are scored on `test`.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
root = next((p for p in [cwd, *cwd.parents] if (p / 'utils' / 'experiments.py').exists()), cwd)
sys.path.insert(0, str(root))

from utils.experiments import DATASETS, RESULTS_DIR
from utils.frames import FRAME_LOADERS
from scripts.run_cv import fold_indices, summarise

pd.set_option('display.width', 200)


## 1. Inspect the folds

Every model sees exactly these partitions.


In [2]:
for key in DATASETS:
    X, y, _ = FRAME_LOADERS[key]()
    sizes = [(len(tr), len(va), len(te)) for _, _, tr, va, te in fold_indices(y, 5, 5)]
    uniq = sorted(set(sizes))
    print(f'{key:10s} n={len(y):6d}  fold sizes: {uniq}  ({len(sizes)} folds)')


compas     n=  7214  fold sizes: [(4616, 1155, 1443), (4617, 1155, 1442)]  (25 folds)
compas_noprot n=  7214  fold sizes: [(4616, 1155, 1443), (4617, 1155, 1442)]  (25 folds)
[Data] Loading German Credit Risk...
german     n=  1000  fold sizes: [(640, 160, 200)]  (25 folds)


adult      n= 45222  fold sizes: [(28941, 7236, 9045), (28942, 7236, 9044)]  (25 folds)
synthetic  n= 20000  fold sizes: [(12800, 3200, 4000)]  (25 folds)


## 2. Run or reload the cross-validation

Full cost on a 6-core CPU + GTX 1650: the baselines take minutes per dataset,
SCORE-Net takes longer because every fold retrains the whole pruning
trajectory. Results are written incrementally, so an interrupted run can be
resumed with the same command.


In [3]:
# Equivalent CLI (recommended for long runs, resumable):
#   python scripts/run_cv.py --models all --datasets german compas compas_noprot adult synthetic

from scripts.run_cv import run_baselines, run_scorenet

RUN = False          # set True to (re)compute instead of loading
DATASETS_TO_RUN = list(DATASETS)
REPEATS, FOLDS = 5, 5

if RUN:
    for key in DATASETS_TO_RUN:
        print(f'--- {key}: SCORE-Net')
        run_scorenet(key, REPEATS, FOLDS)
        print(f'--- {key}: baselines')
        run_baselines(key, REPEATS, FOLDS)


## 3. Mean test accuracy per model

`std` is the sample standard deviation over the 25 folds.


In [4]:
summary = summarise()
summary.round(2)


,dataset,model,variant,n_folds,mean,std
3,COMPAS,CatBoost,SOTA Ensemble,25,67.94,1.17
5,COMPAS,EBM (InterpretML),Interpretable,25,67.94,1.30
2,COMPAS,AdaBoost,SOTA Ensemble,25,67.80,1.34
12,COMPAS,Random Forest,SOTA Ensemble,25,67.59,1.49
0,COMPAS,SCORE-Net,Global Best,25,67.56,1.49
...,...,...,...,...,...,...
55,UCI Adult,MLP (sklearn),Geometric & DL,25,84.73,0.31
49,UCI Adult,Decision Tree,Interpretable,25,84.33,0.43
58,UCI Adult,SVM (RBF),Geometric & DL,25,84.33,0.28
52,UCI Adult,KNN,Geometric & DL,25,82.66,0.35


## 4. Paired comparison against the best baseline

Both models are scored on the *same* fold test sets, so the per-fold
difference is paired. The table reports the mean difference and how often
SCORE-Net wins a fold.


In [5]:
rows = []
for key, spec in DATASETS.items():
    cvs = RESULTS_DIR / f'cv_{key}.csv'
    cvb = RESULTS_DIR / f'cv_baselines_{key}.csv'
    if not (cvs.exists() and cvb.exists()):
        continue
    sn = pd.read_csv(cvs)[['repeat', 'fold', 'test_global', 'test_final']]
    bl = pd.read_csv(cvb)
    # Only folds completed by both are comparable (a run may be in progress).
    bl = bl.merge(sn[['repeat', 'fold']].drop_duplicates(), on=['repeat', 'fold'])
    if bl.empty:
        print(f'{key}: no common folds yet, skipped')
        continue
    best = bl.groupby('model')['test_accuracy'].mean().idxmax()
    bm = bl[bl.model == best][['repeat', 'fold', 'test_accuracy']]
    pair = sn.merge(bm, on=['repeat', 'fold'], how='inner').sort_values(['repeat', 'fold'])
    for col, label in [('test_global', 'Global Best'), ('test_final', 'Compact')]:
        a = pair[col].to_numpy() * 100
        b = pair['test_accuracy'].to_numpy()
        d = a - b
        rows.append({
            'Dataset': spec.display_name, 'Variant': label,
            'Best baseline': best,
            'SCORE-Net': a.mean(), 'Baseline': b.mean(),
            'Delta (pp)': d.mean(), 'Folds won': int((d > 0).sum()), 'Folds': len(d),
        })
pd.DataFrame(rows).round(2)


,Dataset,Variant,Best baseline,SCORE-Net,Baseline,Delta (pp),Folds won,Folds
0,COMPAS,Global Best,CatBoost,67.56,67.94,-0.38,9,25
1,COMPAS,Compact,CatBoost,66.81,67.94,-1.13,5,25
2,COMPAS (no protected attributes),Global Best,EBM (InterpretML),67.67,68.09,-0.43,3,25
3,COMPAS (no protected attributes),Compact,EBM (InterpretML),65.62,68.09,-2.47,2,25
4,German Credit,Global Best,CatBoost,72.72,75.78,-3.06,4,25
5,German Credit,Compact,CatBoost,73.02,75.78,-2.76,2,25
6,UCI Adult,Global Best,LightGBM,85.45,86.97,-1.52,0,25
7,UCI Adult,Compact,LightGBM,85.12,86.97,-1.85,0,25
8,Synthetic,Global Best,MLP (sklearn),95.58,71.96,23.62,25,25
9,Synthetic,Compact,MLP (sklearn),89.32,71.96,17.36,25,25


## 5. Files

* `results/cv_<dataset>.csv` — SCORE-Net, one row per (repeat, fold)
* `results/cv_baselines_<dataset>.csv` — baselines, one row per (repeat, fold, model)
* `results/cv_summary.csv` — the table in section 3


In [6]:
for p in sorted(RESULTS_DIR.glob('cv_*.csv')):
    print(f'{p.name:34s} {len(pd.read_csv(p)):4d} rows')


cv_adult.csv                         25 rows
cv_baselines_adult.csv              325 rows
cv_baselines_compas.csv             325 rows
cv_baselines_compas_noprot.csv      325 rows
cv_baselines_german.csv             325 rows
cv_baselines_synthetic.csv          325 rows
cv_compas.csv                        25 rows
cv_compas_noprot.csv                 25 rows
cv_german.csv                        25 rows
cv_summary.csv                       75 rows
cv_synthetic.csv                     25 rows
